## Loading the collection data from locally to bq
To have the data available on Big Query, we have to do these steps:

- Firstly, we upload the export from the collection file that created the main dataframe, to a bucket in cloud storage (gcs).
- and secondly, from that bucket, we create the data table based on the configuration.

In [ ]:
!pip3 install -r requirements.txt --quiet

In [23]:
from google.cloud import storage
from google.cloud import bigquery
import pandas as pd

In [ ]:
client=bigquery.Client('<your_project_id>')


/Users/Kostas/Documents/Training_discogs_project/.venv/lib/python3.11/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [ ]:
project_id='<your_project_id>' # you can change this to your project name
dataset_id='discogs_db' # you can change this to your dataset name
table_id='my_collection_table' # you can change this to your table name
file_path='data/df_source_translated.csv'

In [ ]:
table_ref=f"{project_id}.{dataset_id}.{table_id}"
print(table_ref)

## Uploading to gcs


In [ ]:
def upload_to_gcs(bucket_name, local_file_path, destination_blob_name):
    client = storage.Client(project='<your_project_id>')
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(local_file_path)
    print(f'Uploaded {local_file_path} to gs://{bucket_name}/{destination_blob_name}')

In [ ]:
upload_to_gcs('<your_bucket_name>','data/df_source_translated.csv','my_discogs_collection.csv')

## Creating a table from gcs

In [63]:
schema=[ bigquery.SchemaField("id", "INTEGER"), 
        bigquery.SchemaField("master_id", "INTEGER"), 
        bigquery.SchemaField("title", "STRING"),
        bigquery.SchemaField("year", "INTEGER"),
        bigquery.SchemaField("genres", "STRING"),
        bigquery.SchemaField("styles", "STRING"),   
        bigquery.SchemaField("date_added", "TIMESTAMP"), 
        bigquery.SchemaField("artists_1_name", "STRING"),
        bigquery.SchemaField("artists_2_name", "STRING"),
        bigquery.SchemaField("artists_1_id", "INTEGER"),
        bigquery.SchemaField("artists_2_id", "INTEGER"),
        bigquery.SchemaField("formats_1_name", "STRING"),
        bigquery.SchemaField("formats_1_qty", "INTEGER"),
        bigquery.SchemaField("formats_1_descriptions_1", "STRING"),
        bigquery.SchemaField("formats_1_descriptions_2", "STRING"),
        bigquery.SchemaField("labels_1_name", "STRING"),
        bigquery.SchemaField("labels_1_id", "INTEGER")                  
              ]

In [ ]:
def load_gcs_to_bigquery(
    gcs_uri: str,
    table_id: str,
    schema: list = None,
    source_format: str = "CSV",
    skip_leading_rows: int = 1,
    write_disposition: str = "WRITE_APPEND",
    autodetect: bool = False,
):
    """
    Load a file from Google Cloud Storage into a BigQuery table.

    Parameters
    ----------
    gcs_uri : str
        Full GCS path, e.g. "gs://bucket/path/file.csv"
    table_id : str
        BigQuery table in format "project.dataset.table"
    schema : list[bigquery.SchemaField], optional
        Explicit schema. Required unless autodetect=True.
    source_format : str
        "CSV", "NEWLINE_DELIMITED_JSON", or "PARQUET"
    skip_leading_rows : int
        Number of header rows to skip (CSV only)
    write_disposition : str
        "WRITE_APPEND", "WRITE_TRUNCATE", or "WRITE_EMPTY"
    autodetect : bool
        Let BigQuery infer schema
    """

    client = bigquery.Client('<your_project_id>')

    # Map string to BigQuery enum
    format_map = {
        "CSV": bigquery.SourceFormat.CSV,
        "JSON": bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        "PARQUET": bigquery.SourceFormat.PARQUET,
    }

    job_config = bigquery.LoadJobConfig(
        source_format=format_map[source_format],
        schema=schema,
        skip_leading_rows=skip_leading_rows if source_format == "CSV" else None,
        write_disposition=write_disposition,
        autodetect=autodetect,
    )

    load_job = client.load_table_from_uri(
        gcs_uri,
        table_id,
        job_config=job_config
    )

    load_job.result()  # Wait for job to finish

    print(f"Loaded data from {gcs_uri} into {table_id}")
    

In [ ]:
load_gcs_to_bigquery(gcs_uri='gs://<your_bucket_name>/my_discogs_collection.csv', 
                     table_id='<your_project_id>.discogs_db.test_discogs', schema=schema,
                     source_format='CSV',autodetect=False)